# 03 — Feature Engineering Audit

The Gold layer (`src/features/gold_enrichment.py`) produces two tables:

- `outlet_monthly.parquet` — outlet × month, with `is_constrained` flag
- `outlet_features.parquet` — wide outlet-level table consumed by the modeling layer

This notebook audits both: are the constraint flags firing the way we'd expect? Are the POI features merged correctly? Do the seasonality-deflated volumes look sensible?

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.utils.io import read_parquet
from src.features.gold_enrichment import run_gold_enrichment

pd.set_option("display.max_columns", 60)

In [ ]:
run_gold_enrichment()
monthly = read_parquet(config.GOLD_DIR / "outlet_monthly.parquet")
features = read_parquet(config.GOLD_DIR / "outlet_features.parquet")
print(f"monthly:  {monthly.shape}")
print(f"features: {features.shape}")
display(monthly.head())
display(features.head())

## 1. Constraint-flag coverage

We want **most** outlets to have at least one unconstrained month (so method 3 can run on them). If `months_unconstrained` is heavily zero, we depend more on peer ceiling and quantile regression — useful to know before modeling.

In [ ]:
cov = pd.DataFrame({
    "constrained_share": [monthly["is_constrained"].mean()],
    "outlets_with_>=1_unc_month": [int((features["months_unconstrained"].fillna(0) >= 1).sum())],
    "outlets_with_>=2_unc_month": [int((features["months_unconstrained"].fillna(0) >= 2).sum())],
    "outlets_total": [len(features)],
})
display(cov)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
features["months_unconstrained"].fillna(0).astype(int).value_counts().sort_index().plot.bar(ax=ax[0])
ax[0].set_title("Outlets by # unconstrained months"); ax[0].set_xlabel("months_unconstrained")

reasons = monthly[["any_credit_cap", "stockout_flag", "soft_ceiling_flag"]].mean()
reasons.plot.bar(ax=ax[1])
ax[1].set_title("Per-month-flag activation rate"); ax[1].set_ylim(0, 1)
plt.tight_layout()

## 2. Seasonality deflation sanity

After dividing by the distributor-month seasonality index, the spread of monthly volume within an outlet should *shrink* (we've removed a known source of variance). A simple plot of CV before vs after confirms this.

In [ ]:
def cv(s):
    s = s.dropna()
    return float(s.std() / s.mean()) if len(s) > 1 and s.mean() > 0 else np.nan

per_outlet = monthly.groupby("outlet_id").agg(
    cv_raw=("volume_total", cv),
    cv_deflated=("volume_total_deflated", cv),
).dropna()
print("median CV raw     :", per_outlet["cv_raw"].median())
print("median CV deflated:", per_outlet["cv_deflated"].median())

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(per_outlet["cv_raw"], bins=40, alpha=0.5, label="raw")
ax.hist(per_outlet["cv_deflated"], bins=40, alpha=0.5, label="deflated")
ax.legend(); ax.set_xlabel("CV"); ax.set_title("Within-outlet monthly volume CV")
plt.tight_layout()

## 3. POI join sanity

We should not see >5% null on POI count columns for outlets with valid coordinates.

In [ ]:
poi_cols = [c for c in features.columns if c.startswith("poi_")]
if poi_cols:
    null_share = features[poi_cols].isna().mean().sort_values(ascending=False)
    print("top POI columns by null share:")
    print(null_share.head(10))
    print(f"\noutlets with no POI features at all: {features[poi_cols].isna().all(axis=1).sum():,}")
else:
    print("no POI feature columns merged in — run the POI scraper or check the join.")